In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim
import optuna

In [21]:
df = pd.read_csv("fmnist_small.csv")
df.head(3)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0


In [22]:
x = df.iloc[:, 1:]
y = df.iloc[:, 0]

In [23]:
X_train, X_test, Y_train, Y_test = train_test_split(x, y, test_size=0.2, random_state=42)
X_train

,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
3897,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5628,0,0,0,0,0,0,0,0,0,1,...,91,0,0,0,0,0,0,0,0,0
1756,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2346,0,0,0,0,0,1,0,0,0,0,...,1,0,0,0,0,65,23,0,0,0
2996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3772,0,0,0,0,0,0,0,0,0,67,...,136,120,73,0,0,0,0,0,0,0
5191,0,0,0,0,0,0,1,0,0,51,...,0,0,1,0,8,66,0,0,0,0
5226,0,0,0,0,0,0,3,1,0,0,...,112,121,121,7,0,1,0,0,0,0
5390,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [24]:
X_train = X_train/255.0
X_test = X_test/255.0

In [25]:
  # Creating Custom Dataset Class
class CustomDataset(Dataset):
    def __init__(self, features, labels):
      self.features = torch.tensor(features, dtype=torch.float32)
      self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
      return len(self.features)

    def __getitem__(self, index):
      return self.features[index], self.labels[index]

In [26]:
# Train dataset object
train_dataset = CustomDataset(X_train.values, Y_train.values)

In [27]:
# Test Dataser object
test_dataset = CustomDataset(X_test.values, Y_test.values)

In [28]:
# Creating Train and TEst Loader
train_loader = DataLoader(train_dataset, batch_size=32, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=32, drop_last=True)

In [29]:
# Define NN Class
class myNN(nn.Module):
    def __init__(self, input_feat, output_feat, hidden_layers, neurons):
        super().__init__()

        layers = []
        prev_feat = input_feat

        # Input Layers
        for i in range(hidden_layers):
            layers.append(nn.Linear(prev_feat, neurons))
            layers.append(nn.BatchNorm1d(neurons)) # Must match output of previous layer
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p=0.3))
            prev_feat = neurons

        # Output Layers
        layers.append(nn.Linear(neurons, output_feat))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

## Optuna Code

In [34]:
def objective(trial):
    input_feat = X_train.shape[1]
    output_feat = 10 # Fashion MNIST has 10 classes
    
    hidden_layers = trial.suggest_int("hidden_layers", 1, 5)
    neurons = trial.suggest_int("neurons", 32, 256)

    model = myNN(input_feat, output_feat, hidden_layers, neurons)

    # Loss Func
    criterion = nn.CrossEntropyLoss()

    # Optimizer
    optimizer = optim.SGD(model.parameters(), lr=0.01, weight_decay=1e-4)

    # Training Loop
    for epochs in range(10):
        model.train()

        for batch_features, batch_labels in train_loader:
            y_pred = model(batch_features)

            # PyTorch expects input (y_pred), target (batch_labels)
            loss = criterion(y_pred, batch_labels)

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()
    
    # Calculate accuracy on testing/validation set
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_features, batch_labels in test_loader:
            outputs = model(batch_features)
            _, predicted = torch.max(outputs, 1)
            total += batch_labels.size(0)
            correct += (predicted == batch_labels).sum().item()

    accuracy = correct / total
    return accuracy


In [35]:
import optuna

# Create an Optuna study to maximize accuracy
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print("Best trial:")
trial = study.best_trial

print(f"  Value (Accuracy): {trial.value:.4f}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")


[I 2026-06-05 11:10:42,569] A new study created in memory with name: no-name-79db2031-dc4c-44f6-b1fd-e13456002b9c
[I 2026-06-05 11:10:45,975] Trial 0 finished with value: 0.8277027027027027 and parameters: {'hidden_layers': 4, 'neurons': 207}. Best is trial 0 with value: 0.8277027027027027.
[I 2026-06-05 11:10:48,780] Trial 1 finished with value: 0.8302364864864865 and parameters: {'hidden_layers': 3, 'neurons': 183}. Best is trial 1 with value: 0.8302364864864865.
[I 2026-06-05 11:10:52,356] Trial 2 finished with value: 0.7837837837837838 and parameters: {'hidden_layers': 5, 'neurons': 124}. Best is trial 1 with value: 0.8302364864864865.
[I 2026-06-05 11:10:55,081] Trial 3 finished with value: 0.8353040540540541 and parameters: {'hidden_layers': 3, 'neurons': 192}. Best is trial 3 with value: 0.8353040540540541.
[I 2026-06-05 11:10:58,936] Trial 4 finished with value: 0.8158783783783784 and parameters: {'hidden_layers': 5, 'neurons': 170}. Best is trial 3 with value: 0.83530405405405

Best trial:
  Value (Accuracy): 0.8395
  Params: 
    hidden_layers: 2
    neurons: 217


In [36]:
best_hidden_layers = study.best_params["hidden_layers"]
best_neurons = study.best_params["neurons"]

input_feat = X_train.shape[1]
output_feat = 10

best_model = myNN(input_feat, output_feat, best_hidden_layers, best_neurons)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(best_model.parameters(), lr=0.01, weight_decay=1e-4)

# Re-train using the best parameters
epoch = 15
for e in range(epoch):
    best_model.train()
    for batch_features, batch_labels in train_loader:
        outputs = best_model(batch_features)
        loss = criterion(outputs, batch_labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


In [38]:
# Calculate Final Test Accuracy
best_model.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        outputs = best_model(batch_features)
        _, predicted = torch.max(outputs, 1)
        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum().item()

final_accuracy = 100 * correct / total
print(f'Final Test Accuracy with Best Params: {final_accuracy:.2f}%')


Final Test Accuracy with Best Params: 83.11%
